[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/infer-actively/pymdp/blob/main/examples/advanced/infer_states_optimization/methods_test_vmp.ipynb)

In [ ]:
import sys
if "google.colab" in sys.modules:
    %pip install "inferactively-pymdp" -q

In [ ]:
import numpy as np
import jax.numpy as jnp
import jax.tree_util as jtu
import jax.experimental.sparse as jsparse
from jax import nn, vmap, jit, block_until_ready
from functools import partial

from pymdp.utils import init_A_and_D_from_spec, generate_agent_specs_from_parameter_sets

# VMP setup (sequence-based inference with trivial B_dependencies)
from pymdp.utils import init_B_from_spec, get_sample_action_seq, get_sample_state, get_obs_seq_from_actions, pad_individual_Bs

# Hybrid
from pymdp.utils import apply_padding_batched
from pymdp.maths import compute_log_likelihoods_padded, deconstruct_lls
from pymdp.algos import run_vmp_hybrid # For hybrid, clustered hybrid, hybrid block and clustered hybrid block

# Clustered hybrid
from pymdp.utils import get_A_dep_clusters, apply_padding_per_cluster
from pymdp.maths import compute_log_likelihoods_per_cluster, deconstruct_log_likelihoods_per_cluster

# Hybrid block
from pymdp.utils import preprocess_A_for_block_diag, concatenate_observations_block_diag
from pymdp.maths import compute_log_likelihoods_block_diag

# Clustered hybrid block
from pymdp.utils import prep_clustered_block_data
from pymdp.maths import compute_log_likelihoods_block_diag_clustered

# End2end padded
from pymdp.utils import apply_A_end2end_padding_batched, apply_obs_end2end_padding_batched
from pymdp.maths import compute_log_likelihood_per_modality_end2end_padded
from pymdp.algos import run_vmp_end2end_padded

# Clustered end2end
from pymdp.utils import apply_A_end2end_padding_per_cluster, apply_obs_end2end_padding_per_cluster
from pymdp.maths import compute_log_likelihoods_end2end_per_cluster
from pymdp.algos import run_vmp_clustered_end2end

In [1]:
# Define coordinated parameter sets
# (num_factors, num_modalities, state_dim_upper_limit, obs_dim_upper_limit, dim_sampling_type, label)
parameter_sets = [
    (5, 5, 5, 5, 'uniform', 'low'),
    (10, 10, 10, 10, 'uniform', 'medium'),
    (25, 25, 25, 25, 'uniform', 'high'),
    # (125, 125, 125, 125, 'uniform', 'extreme'),  # Uncomment to include extreme cases
]

# Generate agent specs without dumping to file
specs = generate_agent_specs_from_parameter_sets(
    parameter_sets,
    num_agents_per_set=1,
    output_file=None  # Don't save to file
)

spec = specs['arbitrary dependencies'][1]
spec

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


{'num_factors': 10,
 'num_modalities': 10,
 'num_states': [5, 9, 5, 8, 8, 5, 2, 5, 6, 5],
 'num_obs': [2, 2, 3, 2, 2, 6, 4, 5, 6, 2],
 'A_dependencies': [[0, 4, 6, 7, 9],
  [1, 3, 8],
  [6, 9],
  [3, 8],
  [3],
  [0, 7, 9],
  [0],
  [2],
  [6],
  [5]],
 'metadata': {'num_factors': 'medium',
  'num_modalities': 'medium',
  'state_dim_upper_limit': 'medium',
  'obs_dim_upper_limit': 'medium',
  'dim_sampling_type': 'uniform'}}

In [ ]:
num_iter = 8
batch_size = 4
T = 8
tau = 1.
A_sparsity_level = None # E.g., 0.8 for 80% sparsity

num_obs = spec['num_obs']
num_states = spec['num_states']
num_controls = [2 for i in range(spec['num_factors'])]
A_dependencies = spec['A_dependencies']

# The optimized VMP routines assume trivial B_dependencies (each factor depends on its own state only)
B_dependencies = [[f] for f in range(spec['num_factors'])]

A, D = init_A_and_D_from_spec(
    num_obs,
    num_states,
    A_dependencies,
    A_sparsity_level=A_sparsity_level,
    batch_size=batch_size
)
B = init_B_from_spec(num_states, num_controls, batch_size=batch_size)

past_actions = get_sample_action_seq(num_controls, T, batch_size=batch_size)
start_state = get_sample_state(num_states, batch_size=batch_size)
obs_seq = get_obs_seq_from_actions(past_actions, start_state, A, A_dependencies, B, B_dependencies)
o_vec = [nn.one_hot(o, num_obs[m]) for m, o in enumerate(obs_seq)]

B_padded = pad_individual_Bs(B)

### Original VMP imported directly from PyMDP

In [ ]:
from pymdp.inference import update_posterior_states

infer_states_orig_pymdp = vmap(
    partial(
        update_posterior_states,
        A_dependencies=A_dependencies,
        B_dependencies=B_dependencies,
        num_iter=num_iter,
        method='vmp'
    )
)

In [2]:
qs = infer_states_orig_pymdp(A, B, o_vec, past_actions, D)
[q.shape for q in qs], qs

([(4, 9, 5),
  (4, 9, 9),
  (4, 9, 5),
  (4, 9, 8),
  (4, 9, 8),
  (4, 9, 5),
  (4, 9, 2),
  (4, 9, 5),
  (4, 9, 6),
  (4, 9, 5)],
 [Array([[[0.1292915 , 0.18693145, 0.09885556, 0.41762125, 0.16730015],
          [0.17185119, 0.2779999 , 0.05775669, 0.24151404, 0.25087813],
          [0.23448874, 0.05233628, 0.20700014, 0.04851925, 0.45765558],
          [0.13505037, 0.04320913, 0.49371397, 0.08963512, 0.23839137],
          [0.18095763, 0.07158826, 0.32005244, 0.16334663, 0.26405498],
          [0.033763  , 0.08465777, 0.04942879, 0.42596036, 0.40619013],
          [0.05496939, 0.05722599, 0.07451323, 0.59027874, 0.22301258],
          [0.17475802, 0.3595561 , 0.02505597, 0.00190917, 0.43872067],
          [0.37240517, 0.13506542, 0.21927299, 0.10532603, 0.16793038]],
  
         [[0.3163604 , 0.2533193 , 0.11985354, 0.05292427, 0.25754246],
          [0.23953591, 0.11340284, 0.56298786, 0.06540234, 0.01867109],
          [0.30447227, 0.21697816, 0.02434554, 0.01161037, 0.44259363],
 

### Hybrid method

In [ ]:
def infer_states_vmp_hybrid(obs_padded, A_padded, D, past_actions, B_padded, A_shapes, num_states, A_dependencies, B_dependencies, num_iter, tau=1.):
    lls_padded = vmap(compute_log_likelihoods_padded, in_axes=(1, None), out_axes=1)(obs_padded, A_padded)
    log_likelihoods = deconstruct_lls(lls_padded, A_shapes, has_time_axis=True)
    return vmap(
        partial(run_vmp_hybrid, num_states=num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
    )(log_likelihoods, D, past_actions, B_padded)

In [ ]:
A_padded = apply_padding_batched(A)
A_shapes = [a.shape for a in A]

if A_sparsity_level is not None:
    A_padded = jsparse.BCOO.fromdense(A_padded, n_batch=1)

# obs preprocessing
obs_padded = apply_padding_batched(o_vec)

In [3]:
qs1 = infer_states_vmp_hybrid(obs_padded, A_padded, D, past_actions, B_padded, A_shapes, num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 9, 5),
  (4, 9, 9),
  (4, 9, 5),
  (4, 9, 8),
  (4, 9, 8),
  (4, 9, 5),
  (4, 9, 2),
  (4, 9, 5),
  (4, 9, 6),
  (4, 9, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.12929145, 0.18693136, 0.09885561, 0.41762146, 0.16730016],
          [0.17185105, 0.27799982, 0.0577567 , 0.24151419, 0.25087819],
          [0.23448843, 0.05233626, 0.20700005, 0.04851922, 0.45765603],
          [0.13505037, 0.04320913, 0.49371397, 0.08963512, 0.23839137],
          [0.1809576 , 0.07158837, 0.3200522 , 0.16334675, 0.26405504],
          [0.03376301, 0.08465774, 0.04942876, 0.42596015, 0.4061903 ],
          [0.05496941, 0.05722595, 0.07451325, 0.5902786 , 0.22301275],
          [0.17475808, 0.35955608, 0.02505599, 0.00190917, 0.43872064],
          [0.37240508,

In [4]:
# JIT
apply_padding_batched_jit = jit(partial(apply_padding_batched))
infer_states_vmp_hybrid_jit = jit(partial(infer_states_vmp_hybrid, A_shapes=A_shapes, num_states=num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau))
obs_padded = apply_padding_batched_jit(o_vec)

qs1 = infer_states_vmp_hybrid_jit(obs_padded, A_padded, D, past_actions, B_padded)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 9, 5),
  (4, 9, 9),
  (4, 9, 5),
  (4, 9, 8),
  (4, 9, 8),
  (4, 9, 5),
  (4, 9, 2),
  (4, 9, 5),
  (4, 9, 6),
  (4, 9, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.12929145, 0.18693136, 0.09885561, 0.41762146, 0.16730016],
          [0.17185105, 0.27799982, 0.0577567 , 0.24151419, 0.25087819],
          [0.23448843, 0.05233626, 0.20700005, 0.04851922, 0.45765603],
          [0.13505037, 0.04320913, 0.49371397, 0.08963512, 0.23839137],
          [0.1809576 , 0.07158837, 0.3200522 , 0.16334675, 0.26405504],
          [0.03376301, 0.08465774, 0.04942876, 0.42596015, 0.4061903 ],
          [0.05496941, 0.05722595, 0.07451325, 0.5902786 , 0.22301275],
          [0.17475808, 0.35955608, 0.02505599, 0.00190917, 0.43872064],
          [0.37240508,

### Clustered hybrid method

In [ ]:
def infer_states_vmp_clustered_hybrid(obs_clusters, A_clusters, D, past_actions, B_padded, c2o_mapping, A_shapes, num_states, A_dependencies, B_dependencies, num_iter, tau=1.):
    ll_clusters = vmap(compute_log_likelihoods_per_cluster, in_axes=(1, None), out_axes=1)(obs_clusters, A_clusters)
    log_likelihoods = deconstruct_log_likelihoods_per_cluster(ll_clusters, A_shapes, c2o_mapping, has_time_axis=True)
    return vmap(
        partial(run_vmp_hybrid, num_states=num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
    )(log_likelihoods, D, past_actions, B_padded)

In [5]:
c2o_mapping = get_A_dep_clusters(A_dependencies)
A_clusters = apply_padding_per_cluster(A, c2o_mapping)
obs_clusters = apply_padding_per_cluster(o_vec, c2o_mapping)

if A_sparsity_level is not None:
    A_clusters = [jsparse.BCOO.fromdense(a) for a in A_clusters]

qs1 = infer_states_vmp_clustered_hybrid(obs_clusters, A_clusters, D, past_actions, B_padded, c2o_mapping, A_shapes, num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 9, 5),
  (4, 9, 9),
  (4, 9, 5),
  (4, 9, 8),
  (4, 9, 8),
  (4, 9, 5),
  (4, 9, 2),
  (4, 9, 5),
  (4, 9, 6),
  (4, 9, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.12929145, 0.18693136, 0.09885561, 0.41762146, 0.16730016],
          [0.17185105, 0.27799982, 0.0577567 , 0.24151419, 0.25087819],
          [0.23448843, 0.05233626, 0.20700005, 0.04851922, 0.45765603],
          [0.13505037, 0.04320913, 0.49371397, 0.08963512, 0.23839137],
          [0.1809576 , 0.07158837, 0.3200522 , 0.16334675, 0.26405504],
          [0.03376301, 0.08465774, 0.04942876, 0.42596015, 0.4061903 ],
          [0.05496941, 0.05722595, 0.07451325, 0.5902786 , 0.22301275],
          [0.17475808, 0.35955608, 0.02505599, 0.00190917, 0.43872064],
          [0.37240508,

### Hybrid Block method

In [ ]:
def infer_states_vmp_hybrid_block(A_big, obs_big, D, past_actions, B_padded, state_shapes, cuts, num_states, A_dependencies, B_dependencies, num_iter, tau=1., use_einsum=False):
    log_likelihoods = vmap(
        partial(compute_log_likelihoods_block_diag, use_einsum=use_einsum), in_axes=(None, 1, None, None), out_axes=1
    )(A_big, obs_big, state_shapes, cuts)
    return vmap(
        partial(run_vmp_hybrid, num_states=num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
    )(log_likelihoods, D, past_actions, B_padded)

In [6]:
# Create a copy with moved axes for block diagonal method (don't modify original A)
A_moveaxis = [jnp.moveaxis(a, 1, -1) for a in A]
# Preprocess A matrices for block diagonal approach
A_big, state_shapes, cuts = preprocess_A_for_block_diag(A_moveaxis)

if A_sparsity_level is not None:
    A_big = jsparse.BCOO.fromdense(A_big, n_batch=1)

obs_big = concatenate_observations_block_diag(o_vec)

qs1 = infer_states_vmp_hybrid_block(A_big, obs_big, D, past_actions, B_padded, state_shapes=state_shapes, cuts=cuts, num_states=num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 9, 5),
  (4, 9, 9),
  (4, 9, 5),
  (4, 9, 8),
  (4, 9, 8),
  (4, 9, 5),
  (4, 9, 2),
  (4, 9, 5),
  (4, 9, 6),
  (4, 9, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.12929145, 0.18693136, 0.09885561, 0.41762146, 0.16730016],
          [0.17185105, 0.27799982, 0.0577567 , 0.24151419, 0.25087819],
          [0.23448843, 0.05233626, 0.20700005, 0.04851922, 0.45765603],
          [0.13505037, 0.04320913, 0.49371397, 0.08963512, 0.23839137],
          [0.1809576 , 0.07158837, 0.3200522 , 0.16334675, 0.26405504],
          [0.03376301, 0.08465774, 0.04942876, 0.42596015, 0.4061903 ],
          [0.05496941, 0.05722595, 0.07451325, 0.5902786 , 0.22301275],
          [0.17475808, 0.35955608, 0.02505599, 0.00190917, 0.43872064],
          [0.37240508,

### Clustered Hybrid Block method

In [ ]:
def infer_states_vmp_clustered_hybrid_block(A_groups, obs_groups, D, past_actions, B_padded, shape_groups, cut_groups, group_mapping, num_states, A_dependencies, B_dependencies, num_iter, tau=1.):
    num_modalities = len(A_dependencies)
    log_likelihoods = vmap(
        compute_log_likelihoods_block_diag_clustered, in_axes=(None, 1, None, None, None, None), out_axes=1
    )(A_groups, obs_groups, shape_groups, cut_groups, group_mapping, num_modalities)
    return vmap(
        partial(run_vmp_hybrid, num_states=num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
    )(log_likelihoods, D, past_actions, B_padded)

In [7]:
# Cluster modalities (host-side preprocessing) and build one block-diagonal system per group
A_block, obs_block, state_shapes_block, cuts_block, group_mapping = prep_clustered_block_data(A, o_vec)

if A_sparsity_level is not None:
    A_block = [jsparse.BCOO.fromdense(a, n_batch=1) for a in A_block]

qs1 = infer_states_vmp_clustered_hybrid_block(A_block, obs_block, D, past_actions, B_padded, state_shapes_block, cuts_block, group_mapping, num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 9, 5),
  (4, 9, 9),
  (4, 9, 5),
  (4, 9, 8),
  (4, 9, 8),
  (4, 9, 5),
  (4, 9, 2),
  (4, 9, 5),
  (4, 9, 6),
  (4, 9, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.12929145, 0.18693136, 0.09885561, 0.41762146, 0.16730016],
          [0.17185105, 0.27799982, 0.0577567 , 0.24151419, 0.25087819],
          [0.23448843, 0.05233626, 0.20700005, 0.04851922, 0.45765603],
          [0.13505037, 0.04320913, 0.49371397, 0.08963512, 0.23839137],
          [0.1809576 , 0.07158837, 0.3200522 , 0.16334675, 0.26405504],
          [0.03376301, 0.08465774, 0.04942876, 0.42596015, 0.4061903 ],
          [0.05496941, 0.05722595, 0.07451325, 0.5902786 , 0.22301275],
          [0.17475808, 0.35955608, 0.02505599, 0.00190917, 0.43872064],
          [0.37240508,

### End2End padded method

In [ ]:
def infer_states_vmp_end2end_padded(obs_padded, A_padded, D, past_actions, B_padded, num_states, A_dependencies, B_dependencies, num_iter, tau=1., sparsity='ll_only'):
    lls_padded = vmap(
        partial(compute_log_likelihood_per_modality_end2end_padded, sparsity=sparsity), in_axes=(2, None)
    )(obs_padded, A_padded)
    return run_vmp_end2end_padded(lls_padded, D, past_actions, B_padded, num_states, A_dependencies, B_dependencies, num_iter=num_iter, tau=tau)

In [8]:
A_padded = apply_A_end2end_padding_batched(A)

if A_sparsity_level is not None:
    A_padded = jsparse.BCOO.fromdense(A_padded)

max_obs_dim = A_padded.shape[2]

# obs preprocessing
obs_padded = apply_obs_end2end_padding_batched(o_vec, max_obs_dim)

qs1 = infer_states_vmp_end2end_padded(obs_padded, A_padded, D, past_actions, B_padded, num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 9, 5),
  (4, 9, 9),
  (4, 9, 5),
  (4, 9, 8),
  (4, 9, 8),
  (4, 9, 5),
  (4, 9, 2),
  (4, 9, 5),
  (4, 9, 6),
  (4, 9, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.12929143, 0.18693152, 0.09885561, 0.41762143, 0.16729999],
          [0.17185113, 0.27799967, 0.0577567 , 0.24151462, 0.25087792],
          [0.2344886 , 0.05233631, 0.20699982, 0.04851926, 0.45765597],
          [0.13505045, 0.04320903, 0.493714  , 0.08963516, 0.23839127],
          [0.18095754, 0.07158829, 0.3200526 , 0.16334672, 0.26405486],
          [0.033763  , 0.08465786, 0.04942883, 0.42595997, 0.40619034],
          [0.05496946, 0.05722595, 0.07451325, 0.5902786 , 0.22301275],
          [0.17475808, 0.35955608, 0.02505597, 0.00190917, 0.43872064],
          [0.3724053 ,

### Clustered End2End method

In [ ]:
def infer_states_vmp_clustered_end2end(obs_clusters, A_clusters, D, past_actions, B_padded, c2o_mapping, c2s_mapping, max_state_dims, num_states, A_dependencies, B_dependencies, num_iter, tau=1., sparsity='ll_only'):
    ll_clusters = vmap(
        partial(compute_log_likelihoods_end2end_per_cluster, sparsity=sparsity), in_axes=(2, None)
    )(obs_clusters, A_clusters)
    return run_vmp_clustered_end2end(ll_clusters, D, past_actions, B_padded, c2o_mapping, c2s_mapping, max_state_dims, num_states, A_dependencies, B_dependencies, num_iter=num_iter, tau=tau)

In [9]:
A_clusters = apply_A_end2end_padding_per_cluster(A, c2o_mapping)
max_obs_dims = [a.shape[2] for a in A_clusters]
max_state_dims = [a.shape[-1] for a in A_clusters]
obs_clusters = apply_obs_end2end_padding_per_cluster(o_vec, c2o_mapping, max_obs_dims)
c2s_mapping = [[s for o in o_list for s in A_dependencies[o]] for o_list in c2o_mapping]

if A_sparsity_level is not None:
    A_clusters = [jsparse.BCOO.fromdense(a) for a in A_clusters]

qs1 = infer_states_vmp_clustered_end2end(obs_clusters, A_clusters, D, past_actions, B_padded, c2o_mapping, c2s_mapping, max_state_dims, num_states, A_dependencies=A_dependencies, B_dependencies=B_dependencies, num_iter=num_iter, tau=tau)
[q1.shape for q1 in qs1], [jnp.allclose(q, q1) for q, q1 in zip(qs, qs1)], qs1

([(4, 9, 5),
  (4, 9, 9),
  (4, 9, 5),
  (4, 9, 8),
  (4, 9, 8),
  (4, 9, 5),
  (4, 9, 2),
  (4, 9, 5),
  (4, 9, 6),
  (4, 9, 5)],
 [Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool),
  Array(True, dtype=bool)],
 [Array([[[0.12929142, 0.18693149, 0.09885567, 0.41762134, 0.16730005],
          [0.1718512 , 0.27799952, 0.05775669, 0.24151449, 0.25087804],
          [0.23448849, 0.05233627, 0.2070001 , 0.04851924, 0.45765594],
          [0.13505064, 0.04320909, 0.49371397, 0.08963512, 0.23839115],
          [0.18095769, 0.07158834, 0.32005253, 0.1633467 , 0.2640548 ],
          [0.03376299, 0.08465784, 0.04942878, 0.4259599 , 0.40619045],
          [0.05496939, 0.05722594, 0.07451323, 0.59027874, 0.22301269],
          [0.17475802, 0.35955593, 0.02505597, 0.00190917, 0.4387209 ],
          [0.3724052 ,